# Đánh giá embedding và mô hình sinh câu hỏi
Notebook này chạy trên Colab để so sánh hai mô hình embedding và hai mô hình sinh câu hỏi tiếng Việt chuyên ngành pháp luật. vis du

In [1]:
%pip install -q transformers sentence-transformers accelerate einops bitsandbytes pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.3 MB/s eta 0:00:00:00:01


In [2]:
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from typing import List, Dict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
DEVICE

✅ Device: cuda
   GPU Name: Tesla T4
   GPU Memory: 15.83 GB


✅ Device: cuda
   GPU Name: Tesla T4
   GPU Memory: 15.83 GB


'cuda'

In [3]:
from huggingface_hub import login
login(token=None)  # Nhập token HF của bạn khi được yêu cầu để tải model private/gated

## Bộ dữ liệu thử nghiệm
Chúng ta dùng 5 đoạn trích ngắn từ luật hộ tịch/hôn nhân để kiểm tra độ chính xác về ngữ nghĩa và 3 truy vấn để đo khả năng matching.

In [4]:
documents = [
    {
        "id": "marriage_age","text": "Điều 8 Luật Hôn nhân và Gia đình quy định nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên mới được kết hôn."
    },
    {
        "id": "marriage_documents","text": "Hồ sơ đăng ký kết hôn gồm tờ khai theo mẫu và giấy tờ chứng minh tình trạng hôn nhân của mỗi bên."
    },
    {
        "id": "marriage_fee","text": "Đăng ký kết hôn tại cấp xã được miễn lệ phí đối với công dân Việt Nam cư trú trong nước."
    },
    {
        "id": "marriage_time","text": "Trong 03 ngày làm việc kể từ ngày nhận đủ hồ sơ hợp lệ, công chức tư pháp hộ tịch phải giải quyết hồ sơ đăng ký kết hôn."
    },
    {
        "id": "marriage_place","text": "Thẩm quyền đăng ký kết hôn thuộc Ủy ban nhân dân cấp xã nơi cư trú của một trong hai bên nam, nữ."
    }
]
queries = [
    {"id": "q_age", "text": "Điều kiện về độ tuổi khi đăng ký kết hôn"},
    {"id": "q_fee", "text": "Có phải nộp lệ phí khi làm thủ tục kết hôn không"},
    {"id": "q_time", "text": "Thời hạn giải quyết hồ sơ đăng ký kết hôn"}
 ]
documents, queries

([{'id': 'marriage_age',
   'text': 'Điều 8 Luật Hôn nhân và Gia đình quy định nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên mới được kết hôn.'},
  {'id': 'marriage_documents',
   'text': 'Hồ sơ đăng ký kết hôn gồm tờ khai theo mẫu và giấy tờ chứng minh tình trạng hôn nhân của mỗi bên.'},
  {'id': 'marriage_fee',
   'text': 'Đăng ký kết hôn tại cấp xã được miễn lệ phí đối với công dân Việt Nam cư trú trong nước.'},
  {'id': 'marriage_time',
   'text': 'Trong 03 ngày làm việc kể từ ngày nhận đủ hồ sơ hợp lệ, công chức tư pháp hộ tịch phải giải quyết hồ sơ đăng ký kết hôn.'},
  {'id': 'marriage_place',
   'text': 'Thẩm quyền đăng ký kết hôn thuộc Ủy ban nhân dân cấp xã nơi cư trú của một trong hai bên nam, nữ.'}],
 [{'id': 'q_age', 'text': 'Điều kiện về độ tuổi khi đăng ký kết hôn'},
  {'id': 'q_fee', 'text': 'Có phải nộp lệ phí khi làm thủ tục kết hôn không'},
  {'id': 'q_time', 'text': 'Thời hạn giải quyết hồ sơ đăng ký kết hôn'}])

In [5]:
from functools import lru_cache
import torch.nn.functional as F
from dataclasses import dataclass
from tabulate import tabulate


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    pooled = torch.sum(last_hidden_state * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
    return pooled


@dataclass
class EmbeddingBackend:
    model_id: str
    mode: str  # 'sentence-transformer' or 'auto'


# CATI-AI model là repo private nên tạm thời loại bỏ khỏi test để tránh treo notebook
embedding_models = {
    "dangvantuan/vietnamese-document-embedding": EmbeddingBackend(
        "dangvantuan/vietnamese-document-embedding", "sentence-transformer"
    )
}


@lru_cache(maxsize=None)
def load_sentence_transformer(model_id):
    print(f"Loading SentenceTransformer: {model_id}")
    return SentenceTransformer(model_id, trust_remote_code=True)


@lru_cache(maxsize=None)
def load_auto_model(model_id):
    print(f"Loading AutoModel embedding: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    return tokenizer, model


def encode_texts(model_id: str, texts: List[str]):
    backend = embedding_models[model_id]
    if backend.mode == "sentence-transformer":
        model = load_sentence_transformer(model_id)
        embeddings = model.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
        return embeddings
    else:
        tokenizer, model = load_auto_model(model_id)
        batch = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**batch)
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            embs = outputs.pooler_output
        else:
            embs = mean_pool(outputs.last_hidden_state, batch.attention_mask)
        embs = F.normalize(embs, p=2, dim=1)
        return embs.cpu()

In [6]:
def build_similarity_frame(model_id: str):
    doc_texts = [d["text"] for d in documents]
    doc_ids = [d["id"] for d in documents]
    query_texts = [q["text"] for q in queries]
    query_ids = [q["id"] for q in queries]
    doc_embs = encode_texts(model_id, doc_texts)
    query_embs = encode_texts(model_id, query_texts)
    cos_doc = (doc_embs @ doc_embs.T).cpu().numpy()
    cos_query_doc = (query_embs @ doc_embs.T).cpu().numpy()
    doc_df = pd.DataFrame(cos_doc, index=doc_ids, columns=doc_ids)
    query_df = pd.DataFrame(cos_query_doc, index=query_ids, columns=doc_ids)
    return doc_df, query_df

doc_results = {}
for model_name in embedding_models.keys():
    try:
        doc_df, query_df = build_similarity_frame(model_name)
        doc_results[model_name] = {"doc_doc": doc_df, "query_doc": query_df}
        print(f"\n=== {model_name} : nội bộ văn bản ===")
        display(doc_df.round(3))
        print(f"=== {model_name} : truy vấn so với văn bản ===")
        display(query_df.round(3))
    except Exception as exc:
        print(f"⚠️ Không thể chạy model {model_name}: {exc}")

Loading SentenceTransformer: dangvantuan/vietnamese-document-embedding


Loading SentenceTransformer: dangvantuan/vietnamese-document-embedding


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading SentenceTransformer: dangvantuan/vietnamese-document-embedding


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


...

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]


=== dangvantuan/vietnamese-document-embedding : nội bộ văn bản ===


,marriage_age,marriage_documents,marriage_fee,marriage_time,marriage_place
marriage_age,1.000,0.449,0.445,0.450,0.586
marriage_documents,0.449,1.000,0.450,0.589,0.720
marriage_fee,0.445,0.450,1.000,0.406,0.547
marriage_time,0.450,0.589,0.406,1.000,0.578
marriage_place,0.586,0.720,0.547,0.578,1.000


=== dangvantuan/vietnamese-document-embedding : truy vấn so với văn bản ===


,marriage_age,marriage_documents,marriage_fee,marriage_time,marriage_place
q_age,0.773,0.620,0.538,0.534,0.684
q_fee,0.455,0.575,0.618,0.491,0.614
q_time,0.532,0.708,0.496,0.747,0.697


## Thử nghiệm mô hình sinh câu trả lời
So sánh hai mô hình mà chúng ta kỳ vọng phù hợp với hệ thống hiện tại (Qwen luật 4B reasoning và Savoxism Qwen 2.5 3B).

In [9]:
question_context = """Điều 8 Luật Hôn nhân và Gia đình 2014: Nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên mới được kết hôn. Việc kết hôn do nam và nữ tự nguyện quyết định và phải đăng ký tại cơ quan nhà nước có thẩm quyền."""
question_prompt = "Tóm tắt yêu cầu về độ tuổi kết hôn theo đoạn trích trên và trả lời dưới dạng gạch đầu dòng."

def load_chat_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    return tokenizer, model

def generate_response(model_id: str, system_prompt: str, user_prompt: str, temperature: float = 0.1, max_new_tokens: int = 256):
    tokenizer, model = load_chat_model(model_id)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.9)
    generation = outputs[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generation, skip_special_tokens=True).strip()
    return text

In [10]:
import time

question_context = """Điều 8 Luật Hôn nhân và Gia đình 2014: Nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên mới được kết hôn. Việc kết hôn do nam và nữ tự nguyện quyết định và phải đăng ký tại cơ quan nhà nước có thẩm quyền."""

question_prompt = "Tóm tắt yêu cầu về độ tuổi kết hôn theo đoạn trích trên và trả lời dưới dạng gạch đầu dòng."

def load_chat_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    return tokenizer, model

def generate_response(model_id: str, system_prompt: str, user_prompt: str, temperature: float = 0.05, max_new_tokens: int = 256):
    try:
        print(f"  ⏳ Loading model {model_id}...")
        tokenizer, model = load_chat_model(model_id)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.9)
        gen_time = time.time() - start
        
        generation = outputs[0][inputs["input_ids"].shape[-1]:]
        text = tokenizer.decode(generation, skip_special_tokens=True).strip()
        tokens_gen = len(generation)
        return text, gen_time, tokens_gen
    except Exception as e:
        return f"❌ Error: {str(e)}", None, None

system_prompt = "Bạn là trợ lý pháp lý, chỉ trả lời dựa trên đoạn trích và phải nêu rõ tuổi tối thiểu của nam, nữ."

# Danh sách LLM để test (từ nhỏ đến lớn, public models)
question_models = [
    ("VietLegalLM-2B-DPO", "mradermacher/VietLegalLM-Qwen-DPO-GGUF"),  # 2B legal
    ("VietTung04-4B-reasoning", "VietTung04/qwen3-4b-legal-reasoning-finetuned"),  # 4B reasoning
    ("Savoxism-3B-QA", "Savoxism/Qwen-2.5-3B-Instruct-Vietnamese-Legal-QA"),  # 3B QA
    ("thangvip-4B-GRPO-v2", "thangvip/qwen3-4b-vietnamese-legal-grpo-phase-2"),  # 4B GRPO
]

question_results = {}
print("=" * 80)
print("🚀 TESTING LLM MODELS FOR LEGAL Q&A")
print("=" * 80)

for display_name, model_id in question_models:
    print(f"\n📌 Model: {display_name}")
    print(f"   Repo: {model_id}")
    answer, gen_time, tokens = generate_response(
        model_id,
        system_prompt,
        f"Đoạn trích:\n{question_context}\n\nYêu cầu: {question_prompt}",
        temperature=0.05
    )
    
    question_results[display_name] = {
        "model_id": model_id,
        "answer": answer,
        "gen_time": gen_time,
        "tokens": tokens
    }
    
    print(f"   ⏱️  Generation time: {gen_time:.2f}s" if gen_time else "   ❌ Failed")
    print(f"   📊 Tokens generated: {tokens}" if tokens else "")
    print(f"   📝 Answer:\n   {answer[:200]}..." if len(str(answer)) > 200 else f"   📝 Answer:\n   {answer}")
    print("-" * 80)

print("\n" + "=" * 80)
print("📊 SUMMARY TABLE")
print("=" * 80)
summary_data = []
for name, result in question_results.items():
    summary_data.append({
        "Model": name,
        "Time (s)": f"{result['gen_time']:.2f}" if result['gen_time'] else "N/A",
        "Tokens": result['tokens'] or "N/A",
        "Status": "✅ OK" if result['gen_time'] else "❌ FAIL"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

🚀 TESTING LLM MODELS FOR LEGAL Q&A

📌 Model: VietLegalLM-2B-DPO
   Repo: mradermacher/VietLegalLM-Qwen-DPO-GGUF
  ⏳ Loading model mradermacher/VietLegalLM-Qwen-DPO-GGUF...
   ❌ Failed

   📝 Answer:
   ❌ Error: expected str, bytes or os.PathLike object, not NoneType
--------------------------------------------------------------------------------

📌 Model: VietTung04-4B-reasoning
   Repo: VietTung04/qwen3-4b-legal-reasoning-finetuned
  ⏳ Loading model VietTung04/qwen3-4b-legal-reasoning-finetuned...
   ❌ Failed

   📝 Answer:
   ❌ Error: expected str, bytes or os.PathLike object, not NoneType
--------------------------------------------------------------------------------

📌 Model: VietTung04-4B-reasoning
   Repo: VietTung04/qwen3-4b-legal-reasoning-finetuned
  ⏳ Loading model VietTung04/qwen3-4b-legal-reasoning-finetuned...


🚀 TESTING LLM MODELS FOR LEGAL Q&A

📌 Model: VietLegalLM-2B-DPO
   Repo: mradermacher/VietLegalLM-Qwen-DPO-GGUF
  ⏳ Loading model mradermacher/VietLegalLM-Qwen-DPO-GGUF...
   ❌ Failed

   📝 Answer:
   ❌ Error: expected str, bytes or os.PathLike object, not NoneType
--------------------------------------------------------------------------------

📌 Model: VietTung04-4B-reasoning
   Repo: VietTung04/qwen3-4b-legal-reasoning-finetuned
  ⏳ Loading model VietTung04/qwen3-4b-legal-reasoning-finetuned...
   ❌ Failed

   📝 Answer:
   ❌ Error: expected str, bytes or os.PathLike object, not NoneType
--------------------------------------------------------------------------------

📌 Model: VietTung04-4B-reasoning
   Repo: VietTung04/qwen3-4b-legal-reasoning-finetuned
  ⏳ Loading model VietTung04/qwen3-4b-legal-reasoning-finetuned...


tokenizer_config.json: 0.00B [00:00, ?B/s]

🚀 TESTING LLM MODELS FOR LEGAL Q&A

📌 Model: VietLegalLM-2B-DPO
   Repo: mradermacher/VietLegalLM-Qwen-DPO-GGUF
  ⏳ Loading model mradermacher/VietLegalLM-Qwen-DPO-GGUF...
   ❌ Failed

   📝 Answer:
   ❌ Error: expected str, bytes or os.PathLike object, not NoneType
--------------------------------------------------------------------------------

📌 Model: VietTung04-4B-reasoning
   Repo: VietTung04/qwen3-4b-legal-reasoning-finetuned
  ⏳ Loading model VietTung04/qwen3-4b-legal-reasoning-finetuned...
   ❌ Failed

   📝 Answer:
   ❌ Error: expected str, bytes or os.PathLike object, not NoneType
--------------------------------------------------------------------------------

📌 Model: VietTung04-4B-reasoning
   Repo: VietTung04/qwen3-4b-legal-reasoning-finetuned
  ⏳ Loading model VietTung04/qwen3-4b-legal-reasoning-finetuned...


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


KeyboardInterrupt: 

## Phân tích chi tiết kết quả

Bảng dưới đây so sánh chất lượng trả lời của từng mô hình:
- **Model**: Tên mô hình
- **Time**: Thời gian sinh câu trả lời (giây)
- **Tokens**: Số tokens được sinh ra
- **Hallucination Check**: Có trích đúng tuổi 18/20 hay không
- **Format**: Có format gạch đầu dòng như yêu cầu không

In [11]:
import re

# Phân tích chất lượng từng mô hình
def analyze_response(text):
    """Kiểm tra chất lượng câu trả lời"""
    analysis = {
        "has_18": "18" in text,
        "has_20": "20" in text,
        "has_bullet": "-" in text or "•" in text,
        "length": len(text),
        "hallucination_risk": "17" in text or "19" in text,  # Sai tuổi
    }
    return analysis

print("\n" + "=" * 80)
print("📋 DETAILED QUALITY ANALYSIS")
print("=" * 80)

quality_data = []
for name, result in question_results.items():
    answer = result['answer']
    analysis = analyze_response(answer)
    
    quality_data.append({
        "Model": name,
        "Has 18yo": "✅" if analysis['has_18'] else "❌",
        "Has 20yo": "✅" if analysis['has_20'] else "❌",
        "Bullet Format": "✅" if analysis['has_bullet'] else "❌",
        "Hallucination": "⚠️ RISK" if analysis['hallucination_risk'] else "✅ SAFE",
        "Length": analysis['length']
    })

quality_df = pd.DataFrame(quality_data)
print(quality_df.to_string(index=False))

print("\n" + "=" * 80)
print("🏆 RECOMMENDATION")
print("=" * 80)
best_models = quality_df[
    (quality_df["Has 18yo"] == "✅") & 
    (quality_df["Has 20yo"] == "✅") & 
    (quality_df["Hallucination"] == "✅ SAFE")
]

if len(best_models) > 0:
    print(f"✅ Models that PASS all checks:")
    for idx, row in best_models.iterrows():
        print(f"   - {row['Model']}")
else:
    print("⚠️ No models pass all checks perfectly. Check answers manually above.")


📋 DETAILED QUALITY ANALYSIS
                  Model Has 18yo Has 20yo Bullet Format Hallucination  Length
     VietLegalLM-2B-DPO        ❌        ❌             ❌        ✅ SAFE      64
VietTung04-4B-reasoning        ✅        ✅             ❌        ✅ SAFE     215
         Savoxism-3B-QA        ✅        ✅             ❌        ✅ SAFE      69

🏆 RECOMMENDATION
✅ Models that PASS all checks:
   - VietTung04-4B-reasoning
   - Savoxism-3B-QA
